# 02 – Exploratory Data Analysis (medical datasets)
**Project 2 – Ensemble learning for disease prediction** (reference: Mahajan et al., 2024)

Goals of this notebook (they feed the *Dataset description* and *Experiments* sections of the report):
1. Check every dataset: size, class balance, missing values, duplicates, data types.
2. Make sure the **positive class = disease** is defined correctly (needed for precision / recall / F1).
3. Look for **data-leakage / too-easy-data signals**: duplicate rows shared by train and test, features that
   predict the target alone, baselines that score ~100 %.
4. Compare our counts with Table 2 of the article.

Run all cells (`Kernel > Restart & Run All`). UCI data downloads itself the first time (internet needed).
Kaggle files must be placed by hand in `data/raw/` (see README).

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from ensemble_disease.config import load_config, set_seed
from ensemble_disease.data import REGISTRY
from ensemble_disease import eda

cfg = load_config(); SEED = cfg["seed"]; set_seed(SEED)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
eda.RESULTS.mkdir(exist_ok=True)
print("seed =", SEED)

ImportError: cannot import name 'eda' from 'ensemble_disease' (C:\Users\LENOVO\Downloads\ensemble_disease_step1\ensemble_disease\src\ensemble_disease\__init__.py)

## 1. Load the datasets
Each dataset is loaded as `(X, y)` with `y = 1` for the disease class.

In [ ]:
data, errors = eda.load_all()
print(f"Loaded {len(data)}/{len(REGISTRY)} datasets:", list(data))
for k, v in errors.items():
    print(f"\n[NOT LOADED] {k}\n   {v}")
if errors:
    print("\n-> Fix the files above (see README, section Data) and re-run. The notebook continues with what is available.")

## 2. Target definition (what is `y = 1`?)

In [ ]:
pd.DataFrame({k: dict(positive_class_rule=v["note"]) for k, v in REGISTRY.items()}).T

## 3. Summary table

In [ ]:
summary = eda.summary_table(data)
summary.to_csv(eda.RESULTS / "eda_summary.csv")
display(summary)

**How to read it.** `imbalance_ratio` = majority / minority; above ~3 accuracy alone is misleading.
`duplicate_rows` are exact copies of another row (features only): a leakage risk when a random split puts copies on both sides.

In [ ]:
eda.plot_class_balance(summary); plt.show()

## 4. Missing values

In [ ]:
display(summary[["n", "missing_cells", "missing_pct", "rows_with_missing"]])
eda.plot_missing(data); plt.show()

## 5. Comparison with the article (Table 2)
Fill `ARTICLE` with the values from Table 2 of the paper (keys must match `n`, `n_features`, `pos`, `neg`),
then re-run the cell. Example: `"D2_heart_cleveland": dict(n=303, n_features=13)`.

In [ ]:
ARTICLE = {
    # "D2_heart_cleveland": dict(n=..., n_features=...),
}
cmp_ = eda.compare_with_article(summary, ARTICLE)
display(cmp_ if len(cmp_) else "ARTICLE is empty: fill it from Table 2 of the paper.")

## 6. Column types and text values
Look for typos, case/whitespace variants (`ckd` vs `ckd\t`), odd codes (`?`, `0` used as missing).

In [ ]:
for name, (X, y) in data.items():
    print(f"\n===== {name}   shape={X.shape}")
    rep = eda.categorical_report(X)
    if len(rep):
        display(rep)
    else:
        print("   (all features numeric)")

## 7. Leakage check A – exact duplicates between train and test

In [ ]:
leak = pd.DataFrame({k: eda.duplicate_leakage(X, y, seed=SEED) for k, (X, y) in data.items()}).T
leak.to_csv(eda.RESULTS / "eda_duplicate_leakage.csv")
display(leak)

`test_rows_with_train_copy_pct` = % of test rows (random stratified 80/20 split, averaged over 50 splits) that have an
identical copy in the training set. Anything clearly above 0 means a memorising model (e.g. a deep tree, a stacking
with a tree meta-learner) gets part of the test set for free. `dup_groups_conflicting_labels` = identical inputs with
different labels (label noise: a ceiling on achievable accuracy).

## 8. Leakage check B – is a single feature almost the target?

In [ ]:
auc_tables = {k: eda.single_feature_auc(X, y) for k, (X, y) in data.items()}
top = pd.DataFrame({k: dict(zip(["top1", "top2", "top3"],
        [f"{r.feature} ({r.strength})" for r in t.head(3).itertuples()])) for k, t in auc_tables.items()}).T
display(top)

`strength = max(AUC, 1-AUC)` of the feature used alone. Above ~0.90 deserves a manual look (is it a consequence of the disease, e.g. a treatment or a diagnosis code?).

## 9. Sanity baselines (default models, 5-fold stratified CV)
*Not* the final protocol: imputation + one-hot encoding are inside the pipeline (fit on train folds only).

In [ ]:
base = pd.DataFrame({k: eda.quick_baselines(X, y, seed=SEED) for k, (X, y) in data.items()}).T
base.to_csv(eda.RESULTS / "eda_baselines.csv")
display(base)

Reading guide: if `forest_acc` is close to `majority_acc` the features carry little signal;
if `tree_acc` and `forest_acc` are ~1.00 the data is (nearly) trivially separable or leaks.
In both cases differences between ensemble methods will be tiny and statistically meaningless.

## 10. Correlations (numeric features)

In [ ]:
for name, (X, y) in data.items():
    eda.plot_corr(X, y, name); plt.show()

## 11. Automatic warnings

In [ ]:
warn = []
for k in data:
    s = summary.loc[k]
    if s.imbalance_ratio >= 3: warn.append((k, f"imbalanced (ratio {s.imbalance_ratio}) -> use F1/AUPRC, stratified splits"))
    if s.missing_pct > 0:      warn.append((k, f"{s.missing_cells} missing cells -> imputation must be inside CV folds"))
    if s.duplicate_rows > 0:   warn.append((k, f"{s.duplicate_rows} duplicate rows ({s.duplicate_pct}%)"))
    if leak.loc[k, "test_rows_with_train_copy_pct"] >= 1: warn.append((k, f"{leak.loc[k,'test_rows_with_train_copy_pct']}% of test rows have a train copy"))
    if s.constant_cols > 0:    warn.append((k, f"{s.constant_cols} constant column(s)"))
    if s.n < 400:              warn.append((k, f"small (n={s.n}) -> high variance, use repeated CV"))
    if auc_tables[k].strength.iloc[0] >= 0.90: warn.append((k, f"feature '{auc_tables[k].feature.iloc[0]}' alone gives strength {auc_tables[k].strength.iloc[0]}"))
    if base.loc[k, "tree_acc"] >= 0.98: warn.append((k, f"a default tree already reaches {base.loc[k,'tree_acc']} accuracy (too easy / leakage?)"))
    if base.loc[k, "forest_acc"] - base.loc[k, "majority_acc"] < 0.05: warn.append((k, "forest barely beats the majority class"))
w = pd.DataFrame(warn, columns=["dataset", "warning"])
w.to_csv(eda.RESULTS / "eda_warnings.csv", index=False)
display(w if len(w) else "No warning.")

## Next
Send back (copy-paste) the outputs of sections **1, 3, 7, 8, 9 and 11**, or the files in `results/` (`eda_*.csv`).
They decide which datasets we keep (target: 9 to 12), how we split (stratified, repeated CV, de-duplication) and what
the first models notebook must control.